# HW01-B — SQL, Latency, and Metabase

The business team does not care that your notebook works. They want a dashboard that opens fast.

Here you connect to shared Postgres, write SQL, measure latency, create a materialized view in your own schema, and build a Metabase dashboard.

## Submission discipline

This is individual work.

Work locally. Push to GitHub. Use the shared server services through URLs and credentials. Do not SSH into the server.

Do not commit `.env`, `.venv/`, passwords.

## Credentials and shared services

Credentials, service URLs, and connection details are provided on the HW page.

Use those exact values. Everyone must work against the same QBC12 database snapshot and the same shared Metabase/Airflow services.

Do not paste credentials into notebook markdown. Do not commit `.env` files. Do not screenshot passwords.


## Useful references

- PostgreSQL `EXPLAIN`: https://www.postgresql.org/docs/current/sql-explain.html
- PostgreSQL using `EXPLAIN`: https://www.postgresql.org/docs/current/using-explain.html
- Metabase questions: https://www.metabase.com/docs/latest/questions/introduction
- Metabase dashboards: https://www.metabase.com/docs/latest/dashboards/introduction

if you cannot open any one of these contact me : Bale (arianaghamohseni, image of a scared chicken), or Telegram (@arianaghamohseni)

## What to avoid

- `select *` in dashboard queries.
- Creating objects in `core`. You do not own `core`.
- Optimizing without runtime measurements.
- Making Metabase run a massive join every time someone opens the dashboard.

In [3]:
import os, re, time
from pathlib import Path
import pandas as pd
from sqlalchemy import create_engine, text

for path in ['sql', 'reports', 'screenshots']:
    Path(path).mkdir(exist_ok=True)

DB_HOST = os.getenv('QBC12_DB_HOST', DB_HOST_)    # this is in the excel file give in Quera SERVERIP
DB_PORT = os.getenv('QBC12_DB_PORT', '32112')
DB_NAME = os.getenv('QBC12_DB_NAME', 'qbc12_airbnb')
DB_USER = os.getenv('QBC12_DB_USER', DB_USER_) or input('DB user: ').strip()
DB_PASSWORD = os.getenv('QBC12_DB_PASSWORD', DB_PASSWORD_) or input('DB password: ').strip()
STUDENT_ID = os.getenv('QBC12_STUDENT_ID',  STUDENT_ID_) or DB_USER.replace('student_', '')

safe_student = re.sub(r'[^a-zA-Z0-9_]', '_', STUDENT_ID.lower())
STUDENT_SCHEMA = f'student_{safe_student}'
engine = create_engine(f'postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}', pool_pre_ping=True)
with engine.begin() as conn:
    conn.execute(text("SET statement_timeout = '30s'"))
    version = conn.execute(text('select version()')).scalar()
STUDENT_SCHEMA, version[:80]

('student_9978',
 'PostgreSQL 16.14 (Debian 16.14-1.pgdg13+1) on x86_64-pc-linux-gnu, compiled by g')

## 1. Inspect before querying

You are not allowed to write the final query blind. Check columns and row counts first.

In [4]:
columns_sql = '''
select table_schema, table_name, column_name, data_type
from information_schema.columns
where table_schema = 'core'
  and table_name in ('listing', 'calendar_day', 'review')
order by table_name, ordinal_position;
'''
pd.read_sql(columns_sql, engine)

,table_schema,table_name,column_name,data_type
0,core,calendar_day,listing_id,bigint
1,core,calendar_day,date,date
2,core,calendar_day,available,boolean
3,core,calendar_day,price,numeric
4,core,calendar_day,adjusted_price,numeric
5,core,calendar_day,minimum_nights,integer
6,core,calendar_day,maximum_nights,integer
7,core,listing,listing_id,bigint
8,core,listing,host_id,bigint
9,core,listing,neighbourhood_id,integer


In [5]:
row_count_sql = '''
select 'core.listing' as table_name, count(*) as rows from core.listing
union all select 'core.calendar_day', count(*) from core.calendar_day
union all select 'core.review', count(*) from core.review;
'''
pd.read_sql(row_count_sql, engine)

,table_name,rows
0,core.listing,10480
1,core.calendar_day,3825200
2,core.review,501084


## 2. Create your sandbox schema

This is the only place you write database objects.

In [6]:
# TODO 2.1
# Create your schema if it does not exist.
# Schema name is STUDENT_SCHEMA.

# Write your code here.

from sqlalchemy import text

query = text('''
SELECT schema_name 
FROM information_schema.schemata
WHERE schema_name LIKE 'student_%'
''')

STUDENT_SCHEMA = pd.read_sql(query, engine).iloc[0]['schema_name']


## 3. Build baseline SQL in pieces

Do not write one giant query first. Build the CTEs, test them, then combine.

In [8]:
# TODO 3.1
# Write calendar_30_sql.
# Required output: listing_id, avg_calendar_price_30, availability_30_rate.

calendar_30_sql = """
select
    cd.listing_id,
    avg(l.listing_price) as avg_calendar_price_30,
    avg(case when cd.available then 1.0 else 0.0 end) as availability_30_rate
from core.calendar_day cd
join core.listing l
    on cd.listing_id = l.listing_id
where cd.date >= (select min(date) from core.calendar_day)
  and cd.date < (select min(date) from core.calendar_day) + interval '30 days'
group by cd.listing_id
"""

pd.read_sql(calendar_30_sql, engine).head()

,listing_id,avg_calendar_price_30,availability_30_rate
0,1443670960781261954,146.0,0.000000
1,896043282611946316,NaN,0.000000
2,958726726744532841,NaN,0.033333
3,39969190,NaN,0.000000
4,1476051889347548382,214.0,0.200000


In [9]:
# TODO 3.2
# Write review_counts_sql.
# Required output: listing_id, total_reviews.

review_counts_sql = """
select
    listing_id,
    count(*) as total_reviews
from core.review
group by listing_id
"""
pd.read_sql(review_counts_sql, engine).head()

,listing_id,total_reviews
0,1443670960781261954,5
1,896043282611946316,2
2,958726726744532841,23
3,39969190,10
4,1476051889347548382,2


In [10]:
# TODO 3.3
# Combine the CTEs with core.listing into baseline_sql.
# Required output:
# neighbourhood, num_listings, avg_price, median_price,
# avg_minimum_nights, total_reviews, reviews_per_listing, availability_30_rate. 
baseline_sql = """
with calendar_30 as (
    select
        cd.listing_id,
        avg(l.listing_price) as avg_calendar_price_30,
        avg(case when cd.available then 1.0 else 0.0 end) as availability_30_rate
    from core.calendar_day cd
    join core.listing l
        on cd.listing_id = l.listing_id
    where cd.date >= (select min(date) from core.calendar_day)
      and cd.date < (select min(date) from core.calendar_day) + interval '30 days'
    group by cd.listing_id
),

review_counts as (
    select
        listing_id,
        count(*) as total_reviews
    from core.review
    group by listing_id
),

listing_enriched as (
    select
        l.listing_id,
        l.neighbourhood_id as neighbourhood,
        l.listing_price,
        l.minimum_nights,
        coalesce(r.total_reviews, 0) as total_reviews,
        c.availability_30_rate
    from core.listing l
    left join review_counts r
        on l.listing_id = r.listing_id
    left join calendar_30 c
        on l.listing_id = c.listing_id
)

select
    neighbourhood,
    count(*) as num_listings,
    avg(listing_price) as avg_price,
    percentile_cont(0.5) within group (order by listing_price) as median_price,
    avg(minimum_nights) as avg_minimum_nights,
    sum(total_reviews) as total_reviews,
    sum(total_reviews)::float / count(*) as reviews_per_listing,
    avg(availability_30_rate) as availability_30_rate
from listing_enriched
group by neighbourhood
order by neighbourhood
"""

Path("sql/01_baseline_neighbourhood_summary.sql").write_text(baseline_sql)
pd.read_sql(baseline_sql, engine).head()

,neighbourhood,num_listings,avg_price,median_price,avg_minimum_nights,total_reviews,reviews_per_listing,availability_30_rate
0,1,923,307.724138,240.0,4.888407,76899.0,83.314193,0.221416
1,2,1207,315.880519,245.5,4.009114,106496.0,88.231980,0.218862
2,3,436,216.597403,195.0,4.399083,13988.0,32.082569,0.126606
3,4,736,255.040816,214.5,4.240489,26668.0,36.233696,0.139040
4,5,206,193.240000,185.5,3.359223,8725.0,42.354369,0.131068


In [11]:
def timed_read_sql(sql: str, repeats: int = 3):
    times = []
    last_df = None
    for _ in range(repeats):
        start = time.perf_counter()
        last_df = pd.read_sql(sql, engine)
        times.append(time.perf_counter() - start)
    return last_df, times

baseline_df, baseline_times = timed_read_sql(baseline_sql, repeats=3)
baseline_df.head(), baseline_times

(   neighbourhood  num_listings   avg_price  median_price  avg_minimum_nights  \
 0              1           923  307.724138         240.0            4.888407   
 1              2          1207  315.880519         245.5            4.009114   
 2              3           436  216.597403         195.0            4.399083   
 3              4           736  255.040816         214.5            4.240489   
 4              5           206  193.240000         185.5            3.359223   
 
    total_reviews  reviews_per_listing  availability_30_rate  
 0        76899.0            83.314193              0.221416  
 1       106496.0            88.231980              0.218862  
 2        13988.0            32.082569              0.126606  
 3        26668.0            36.233696              0.139040  
 4         8725.0            42.354369              0.131068  ,
 [0.38468859999920824, 0.3642526000003272, 0.37075890000051004])

## 4. Read the query plan

`EXPLAIN ANALYZE` actually runs the query. Look for big scans, expensive joins, and repeated work.

In [ ]:
# TODO 4.1
# Run EXPLAIN (ANALYZE, BUFFERS, FORMAT TEXT) on baseline_sql.
# Save the plan to reports/baseline_explain_analyze.txt.

# Write your code here.
from pathlib import Path

explain_sql = f"""
EXPLAIN (ANALYZE, BUFFERS, FORMAT TEXT)
{baseline_sql}
"""

plan_df = pd.read_sql(explain_sql, engine)

plan_text = "\n".join(plan_df.iloc[:, 0].astype(str))

Path("reports").mkdir(exist_ok=True)
Path("reports/baseline_explain_analyze.txt").write_text(plan_text, encoding="utf-8")

print(plan_text[:3000])




GroupAggregate  (cost=40248.78..40485.24 rows=22 width=156) (actual time=401.157..406.881 rows=22 loops=1)
  Group Key: l.neighbourhood_id
  Buffers: shared hit=13394 read=7736
  ->  Sort  (cost=40248.78..40274.98 rows=10480 width=53) (actual time=400.552..401.487 rows=10480 loops=1)
        Sort Key: l.neighbourhood_id
        Sort Method: quicksort  Memory: 938kB
        Buffers: shared hit=13394 read=7736
        ->  Hash Left Join  (cost=39199.12..39548.96 rows=10480 width=53) (actual time=385.926..396.830 rows=10480 loops=1)
              Hash Cond: (l.listing_id = c.listing_id)
              Buffers: shared hit=13394 read=7736
              ->  Hash Left Join  (cost=13726.93..14049.25 rows=10480 width=29) (actual time=124.963..132.658 rows=10480 loops=1)
                    Hash Cond: (l.listing_id = r.listing_id)
                    Buffers: shared hit=670 read=7736
                    ->  Seq Scan on listing l  (cost=0.00..294.80 rows=10480 width=21) (actual time=0.035..1.526 r

In [13]:
# TODO 4.2
# Write reports/explain_notes.md with 3 specific observations from the plan.
# Do not write vague nonsense like 'the query is slow'.

# Write your code here.

from pathlib import Path

plan_text = Path("reports/baseline_explain_analyze.txt").read_text(encoding="utf-8")

explain_notes = f"""# Explain Analyze Notes

## Observation 1

The query performs a sequential scan on the listing table.

## Observation 2

The review table is grouped by listing_id before joining.

## Observation 3

The final result is grouped by neighbourhood_id to produce neighbourhood-level metrics.

## Raw plan excerpt

```text
{plan_text[:3000]}
"""

Path("reports/explain_notes.md").write_text(explain_notes, encoding="utf-8")



3317

## 5. Create a materialized view

Metabase should read from a prepared object, not a fresh monster join.

In [14]:
# TODO 5.1
# Create optimized_sql.
# It should create student_<you>.mv_airbnb_neighbourhood_summary and at least two indexes.

optimized_sql = f"""
drop materialized view if exists "{STUDENT_SCHEMA}".mv_airbnb_neighbourhood_summary;

create materialized view "{STUDENT_SCHEMA}".mv_airbnb_neighbourhood_summary as
{baseline_sql};

create index idx_mv_airbnb_neighbourhood
on "{STUDENT_SCHEMA}".mv_airbnb_neighbourhood_summary (neighbourhood);

create index idx_mv_airbnb_num_listings
on "{STUDENT_SCHEMA}".mv_airbnb_neighbourhood_summary (num_listings);
"""

Path("sql/02_create_materialized_view.sql").write_text(optimized_sql)

1858

In [15]:
# TODO 5.2
# Execute optimized_sql statement by statement.

# Write your code here.

from sqlalchemy import text

statements = [s.strip() for s in optimized_sql.split(";") if s.strip()]

with engine.begin() as conn:
    for statement in statements:
        conn.execute(text(statement))

In [30]:
check_sql = f'''select * from "{STUDENT_SCHEMA}".mv_airbnb_neighbourhood_summary order by num_listings desc limit 10;'''
pd.read_sql(check_sql, engine)

,neighbourhood,num_listings,avg_price,median_price,avg_minimum_nights,total_reviews,reviews_per_listing,availability_30_rate
0,22,1808,271.282258,240.0,3.949668,62753.0,34.708518,0.155439
1,2,1207,315.880519,245.5,4.009114,106496.0,88.231980,0.218862
2,20,1199,280.465331,250.0,5.464554,45931.0,38.307756,0.170336
3,1,923,307.724138,240.0,4.888407,76899.0,83.314193,0.221416
4,4,736,255.040816,214.5,4.240489,26668.0,36.233696,0.139040
5,21,735,309.583908,238.0,3.993197,29444.0,40.059864,0.192744
6,13,654,251.659509,222.0,4.061162,20448.0,31.266055,0.143986
7,14,547,204.674330,184.0,6.076782,16461.0,30.093236,0.128580
8,18,485,370.496032,196.5,3.336082,22623.0,46.645361,0.164399
9,3,436,216.597403,195.0,4.399083,13988.0,32.082569,0.126606


## 6. Compare latency

Numbers or it did not happen.

In [16]:
dashboard_sql = f"""
select
    neighbourhood,
    num_listings,
    avg_price,
    median_price,
    total_reviews,
    reviews_per_listing,
    availability_30_rate
from "{STUDENT_SCHEMA}".mv_airbnb_neighbourhood_summary
order by num_listings desc;
"""

dashboard_df, dashboard_times = timed_read_sql(dashboard_sql, repeats=5)

perf = pd.DataFrame([
    {
        "query": "baseline_direct_query",
        "best_seconds": min(baseline_times),
        "avg_seconds": sum(baseline_times) / len(baseline_times),
    },
    {
        "query": "materialized_view_read",
        "best_seconds": min(dashboard_times),
        "avg_seconds": sum(dashboard_times) / len(dashboard_times),
    },
])

perf["speedup_vs_baseline_best"] = perf.loc[0, "best_seconds"] / perf["best_seconds"]
perf

,query,best_seconds,avg_seconds,speedup_vs_baseline_best
0,baseline_direct_query,0.364253,0.373233,1.000000
1,materialized_view_read,0.078090,0.079472,4.664547


## 7. Metabase dashboard

Open the shared Metabase URL and create:

```text
QBC12 HW01 - <your-github-username> - Airbnb Ops
```

Required cards:

1. listings by neighbourhood
2. average price by neighbourhood
3. review activity by neighbourhood
4. availability rate by neighbourhood
5. top neighbourhoods table

Screenshot path:

```text
screenshots/metabase_dashboard.png
```

In [ ]:
# TODO 7.1
# Write reports/hw01_b_sql_performance.md.
# Include schema, runtimes, speedup, what changed, and Metabase screenshot/link.

# Write your code here.

In [ ]:
for file in ['sql/01_baseline_neighbourhood_summary.sql','sql/02_create_materialized_view.sql','reports/baseline_explain_analyze.txt','reports/explain_notes.md','reports/hw01_b_sql_performance.md']:
    assert Path(file).exists(), f'Missing {file}'
assert len(dashboard_df) > 0
perf

## Commit

```bash
git add sql reports screenshots notebooks
git commit -m "HW01-B SQL performance and Metabase dashboard"
```